<a href="https://colab.research.google.com/github/zillioxmatthewprojects-ops/Bus_Route_ML/blob/main/Bus_ML_Logistic_Regression_Profit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load libraries and datasets, then test they are loaded.

In [1]:
#Created by Matt Z
#Starting Line up
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from pandas.core.arrays import categorical
from itertools import combinations

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

#import Cleaned bus data
df = pd.read_csv(r"/content/Bus Depot ML2_data.csv")
print("Jumanji 1: Bus Data Loaded")

#import Price Sheet
df_Fare = pd.read_excel(r"/content/Bus ML Fare list.xlsx")
print("Jumanji 2: Bus Price Sheet Data Loaded")
df.info()
df_Fare.head()

Jumanji 1: Bus Data Loaded
Jumanji 2: Bus Price Sheet Data Loaded
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   TripID       25000 non-null  object
 1   TripDate     25000 non-null  object
 2   State        25000 non-null  object
 3   RouteName    25000 non-null  object
 4   VehicleType  25000 non-null  object
 5   Ridership    25000 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 1.1+ MB


,VehicleType,Capacity,Fare,Trip Cost,Break-even riders
0,Mini,35,3.0,70,24
1,Standard,70,3.0,110,37
2,Volt,85,2.5,80,32
3,HellCat,105,2.0,120,60


In [2]:
#add price sheet dataset to main df with merge to create 3 columns with data based on Vehicle and VehicleType
lookup_df = pd.DataFrame(df_Fare, columns=['VehicleType', 'Capacity', 'Fare', 'Trip Cost', 'Break-even riders'])
df = df.merge(
    lookup_df,
    left_on='VehicleType',
    right_on='VehicleType',
    how='left'
)
print("Jumanji 1: Merg Complete\n")

#add new revenue column from trip cost * ridership
df['Trip Revenue'] = df['Fare'] * df['Ridership']
print("Jumanji 2: Add Revenue Complete\n")

#add new profit column from trip revenue - trip cost
df['Trip Profit'] = df['Trip Revenue'] - df['Trip Cost']
print("Jumanji 3: Add Profit Complete\n")
df.head()

#Add Binary column is this ride profitable (Ridership > Break-even Riders)
df['Profitable Trip'] = np.where(df['Ridership'] > (df['Break-even riders']+1), 'Yes', 'No')

#add trip date info columns
df['TripDate'] = pd.to_datetime(df['TripDate'])
df['Month_num'] = df['TripDate'].dt.month
df.head()

Jumanji 1: Merg Complete

Jumanji 2: Add Revenue Complete

Jumanji 3: Add Profit Complete



,TripID,TripDate,State,RouteName,VehicleType,Ridership,Capacity,Fare,Trip Cost,Break-even riders,Trip Revenue,Trip Profit,Profitable Trip,Month_num
0,TRIP-16868,2023-02-12,New York,Route 6,Standard,29,70,3.0,110,37,87.0,-23.0,No,2
1,TRIP-34016,2023-10-04,New York,Route 5,Mini,27,35,3.0,70,24,81.0,11.0,Yes,10
2,TRIP-19668,2023-07-14,California,Route 1,Standard,40,70,3.0,110,37,120.0,10.0,Yes,7
3,TRIP-23640,2023-09-08,California,Route 1,Volt,47,85,2.5,80,32,117.5,37.5,Yes,9
4,TRIP-24018,2022-07-24,Texas,Route 3,HellCat,26,105,2.0,120,60,52.0,-68.0,No,7


Feature encoding.

In [3]:
# Feature Encoding
#Declare variables, encode, scale numbers, split data
target_variable = ['Profitable Trip']
df['Profitable Trip'] = df['Profitable Trip'].map({'Yes': 1, 'No': 0})
print(df['Profitable Trip'].unique())  # should show [0 1] now

categorical_features = ['VehicleType', 'RouteName', 'State']
numeric_features = ['Month_num']

predictors = ['VehicleType', 'RouteName', 'State', 'Month_num']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print()
print(preprocessor)

[0 1]
Numeric features: ['Month_num']
Categorical features: ['VehicleType', 'RouteName', 'State']

ColumnTransformer(transformers=[('num', StandardScaler(), ['Month_num']),
                                ('cat', OneHotEncoder(drop='first'),
                                 ['VehicleType', 'RouteName', 'State'])])


Split data for ML and create Pipeline

In [4]:
# Train / Validation / Test split 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    df[predictors], df[target_variable], test_size=0.30, random_state=28
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=28
)

# Verification
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Pipeline
# Bundle preprocessing + model into one object
bus_pipeline = Pipeline([
    ('scaler_encoder', preprocessor),
    ('logreg', LogisticRegression(
        solver='saga',
        max_iter=5000,
        class_weight='balanced'
    ))
])
print("Pipeline complete")

Train: 17500 | Val: 3750 | Test: 3750
Pipeline complete


Perform Logistic Regression

In [5]:
# Logistic Regression Model

# 1 Define parameters
param_grid = {
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__penalty': ['l1', 'l2']
}

# 2 Try all combinations using cross-validation
grid_search = GridSearchCV(bus_pipeline, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train.values.ravel())

print(f"Best Params: {grid_search.best_params_}\n")

# 3 Analysis Classification Report
print("Initial Model")
y_pred = grid_search.predict(X_val)
print(classification_report(y_val, y_pred))

Best Params: {'logreg__C': 0.01, 'logreg__penalty': 'l2'}

Initial Model
              precision    recall  f1-score   support

           0       0.82      0.60      0.69      2362
           1       0.53      0.77      0.63      1388

    accuracy                           0.66      3750
   macro avg       0.67      0.69      0.66      3750
weighted avg       0.71      0.66      0.67      3750

